In [ ]:
import os
import torch
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

from detection import *

In [ ]:
device = get_device()
device

In [ ]:
hparams = {
    'sigma': 3,
    'optimizer': 'AdamW',
    'lr': 5e-4,
    'weight_decay': 5e-4,
    'batch_size': 32,
    'img_size': '100x100',
    'scheduler': 'OneCycleLR',
    'max_lr': 5e-3,
    'ch_mul': 32,
    'grad_clip': None,
    'epochs': 50,
    'log_dir': './logs/exp_model_heatmap'
}

In [ ]:
data_path = './data/'
train_img_dir = os.path.join(data_path, 'images')
train_gt = os.path.join(data_path, 'gt.csv')

In [ ]:
train_loader, val_loader = prepare_dataloaders(
    image_dir=train_img_dir,
    gt=train_gt,
    img_size=tuple(map(int, hparams['img_size'].split('x'))),
    split=(0.9, 0.1),
    batch_size=hparams['batch_size'],
    num_workers=0,
    sigma=hparams['sigma'],
)

In [ ]:
model = UNet(ch_mul=hparams['ch_mul'])
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hparams['lr'],
    weight_decay=hparams['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=hparams['max_lr'],
    steps_per_epoch=len(train_loader),
    epochs=hparams['epochs'],
)
logger = SummaryWriter(log_dir=hparams['log_dir'])
trainer = Trainer(
    model=model.to(device),
    criterion=weighted_mse_loss,
    optimizer=optimizer,
    scheduler=scheduler,
    logger=logger,
    device=device,
)
sum(p.numel() for p in model.parameters())

In [ ]:
val_losses = trainer.train(
    train_loader, val_loader, n_epochs=hparams['epochs'],
)
logger.add_hparams(hparams, {"val_loss": val_losses[-1]})
logger.close()

In [ ]:
torch.save(
    {
        'epoch': hparams['epochs'],
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': val_losses[-1],
    },
    './checkpoints/pretrained_model_checkpoint_sigma3.pt',
)